In [ ]:
import os

dataset_path = "/kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset"

print(os.listdir(dataset_path))

['fashion-dataset']


In [13]:
import pandas as pd

styles = pd.read_csv(
    "/kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset/fashion-dataset/styles.csv",
    on_bad_lines = "skip"
)

styles.head()

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [14]:
!pip install sentence-transformers scikit-learn pandas

In [15]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

from sklearn.cluster import DBSCAN

In [16]:
import pandas as pd

df = pd.read_csv(
    "/kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset/fashion-dataset/styles.csv",
    on_bad_lines = "skip"
)

df.head()

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [17]:
print(df.columns.tolist())

['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName']


In [18]:
df = df[[
    'id',
    'productDisplayName',
    'gender',
    'masterCategory',
    'subCategory',
    'articleType',
    'baseColour',
    'season'
]]

df = df.dropna()

In [19]:
df["text"] = (
    df["productDisplayName"].astype(str) + " " +
    df["articleType"].astype(str) + " " +
    df["baseColour"].astype(str) + " " +
    df["gender"].astype(str) + " " +
    df["subCategory"].astype(str)
)

In [21]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df["text"].tolist(),
    show_progress_bar=True
)

'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Batches:   0%|          | 0/1388 [00:00<?, ?it/s]

In [23]:
from sklearn.cluster import DBSCAN

cluster = DBSCAN(
    eps=0.18,
    min_samples=1,
    metric="cosine"
)

labels = cluster.fit_predict(embeddings)

df["Cluster"] = labels

In [26]:
catalog = (
    df.groupby("Cluster")
      .first()
      .reset_index()
)

catalog = catalog[[
    "productDisplayName",
    "articleType",
    "baseColour",
    "gender"
]]

catalog.head()

,productDisplayName,articleType,baseColour,gender
0,Manchester United Men Solid Grey Track Pants,Track Pants,Grey,Men
1,Fastrack Women Steel Dial Watch,Watches,Steel,Women
2,French Connection Men Brown Belt,Belts,Brown,Men
3,Murcia Women Casual Purple Handbag,Handbags,Purple,Women
4,Playboy Women Pack of 3 Deos,Deodorant,Black,Women


In [27]:
df = df.sample(1000, random_state=42).reset_index(drop=True)

In [28]:
df["text"] = (
    df["productDisplayName"].astype(str)
    + " "
    + df["articleType"].astype(str)
    + " "
    + df["baseColour"].astype(str)
)

In [29]:
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [30]:
embeddings = model.encode(
    df["text"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [31]:
from sklearn.cluster import DBSCAN

cluster = DBSCAN(
    eps=0.25,
    min_samples=1,
    metric="cosine"
)

labels = cluster.fit_predict(embeddings)

df["Cluster"] = labels

In [32]:
labels = cluster.fit_predict(embeddings)
df["Cluster"] = labels

In [33]:
duplicate_groups = df.groupby("Cluster")

for cluster_id, group in duplicate_groups:
    if len(group) > 1:
        print("=" * 80)
        print(f"Cluster {cluster_id}")
        print(group["productDisplayName"].tolist())

Cluster 3
['Myntra Women Yellow T-shirt', 'Myntra Women Blue and Yellow Printed Top']
Cluster 4
['Nike Womens Luck White T-shirt', 'Nike Men As Elite Ultimate Ss Training Purple T-Shirts', 'ADIDAS Black CC Oscillate M Sports Shoe', 'Vans Unisex Authentic Black & Green Shoes', 'French Connection Men White Printed T-shirt', 'Wrangler Women Big Rodeo Pink T-shirt', 'Basics Men Blue & White Slim Fit Checked Shirt', 'French Connection Men Blue T-shirt', 'Rockport Men Schemerhorn Black Shoe', 'Gas Men Bugey Blue Shoes', "Red Tape Men's Casual Grey Shoe", 'Fila Mens Ankel Logo White Socks', 'Puma Men Diva Fashion Black Sandal', 'Catwalk Women Black Heels', 'Nike Men Court Majestic Black Sports Shoes', 'Lotto Men White Sports Shoes', 'Reebok Men Speed Mode White Sports Shoes', 'Nike Men Grey Striped T-shirt', 'Nike Womens Air Rele Purple Shoe', 'U.S. Polo Assn. Men White T-shirt', 'United Colors of Benetton Men Solid Blue Shirts', 'Catwalk Women Orange Flats', "Carlton London Men's Semi Formal

In [34]:
catalog = (
    df.groupby("Cluster")
      .first()
      .reset_index()
)

In [35]:
catalog = catalog[
    [
        "productDisplayName",
        "articleType",
        "baseColour"
    ]
]

In [36]:
print(catalog.head(20))

                                   productDisplayName            articleType  \
0   Superman Men's Superman Shield Desi Dark Grey ...                Tshirts   
1   Shree Women Black & White Printed Salwar Suit ...             Kurta Sets   
2                                  Avirate Purple Bra                    Bra   
3                         Myntra Women Yellow T-shirt                Tshirts   
4                      Nike Womens Luck White T-shirt                Tshirts   
5                    Wrangler Men Blue Skanders Jeans                  Jeans   
6          Puma Men Top Fluctuation Red Black Watches                Watches   
7              Jockey Women Grey Melange T-shirt 1359                Tshirts   
8                ADIDAS Men Dumont Beige Sports Shoes           Sports Shoes   
9              Fastrack Women Campus Brown Sunglasses             Sunglasses   
10                Baldessarini Men Concentree Perfume  Perfume and Body Mist   
11                      Murcia Women Cha

In [37]:
catalog.to_csv("Unique_Product_Catalog.csv", index=False)

print("Unique Product Catalog Saved Successfully!")

Unique Product Catalog Saved Successfully!


In [38]:
from IPython.display import FileLink

FileLink("Unique_Product_Catalog.csv")

/kaggle/working/Unique_Product_Catalog.csv

In [39]:
import pandas as pd
import numpy as np
import re

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ============================
# Load Dataset
# ============================

df = pd.read_csv("/kaggle/input/datasets/jaiwantkarrun/sample-product/product.csv")

print("="*60)
print("INPUT DATASET")
print("="*60)

for product in df["ProductName"]:
    print(product)

# ============================
# Load Embedding Model
# ============================

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# ============================
# Generate Embeddings
# ============================

embeddings = model.encode(
    df["ProductName"].tolist(),
    convert_to_numpy=True
)

# ============================
# Cosine Similarity
# ============================

similarity = cosine_similarity(embeddings)

# Similarity threshold
threshold = 0.75

visited = set()
groups = []

# ============================
# Create Duplicate Groups
# ============================

for i in range(len(df)):

    if i in visited:
        continue

    group = [i]
    visited.add(i)

    for j in range(i + 1, len(df)):

        if similarity[i][j] >= threshold:
            group.append(j)
            visited.add(j)

    groups.append(group)

# ============================
# Function to Generate
# Representative Product Name
# ============================

def representative_name(products):

    first = products[0]

    # Remove A,B,C at end
    first = re.sub(r"\s+[A-Z]$", "", first)

    # Remove extra spaces
    first = " ".join(first.split())

    return first

# ============================
# Display Duplicate Groups
# ============================

catalog = []

print("\n")
print("="*60)
print("DUPLICATE PRODUCT GROUPS")
print("="*60)

group_no = 1

for group in groups:

    products = df.iloc[group]["ProductName"].tolist()

    if len(products) > 1:

        print(f"\nGroup {group_no}")
        print("-"*40)

        for p in products:
            print(p)

        rep = representative_name(products)

        print("\nRepresentative Product:")
        print(rep)

        catalog.append(rep)

        group_no += 1

# ============================
# Add Unique Products
# ============================

for group in groups:

    products = df.iloc[group]["ProductName"].tolist()

    if len(products) == 1:

        catalog.append(products[0])

# Remove duplicates
catalog = list(dict.fromkeys(catalog))

# ============================
# Final Catalog
# ============================

print("\n")
print("="*60)
print("FINAL UNIQUE PRODUCT CATALOG")
print("="*60)

for i, product in enumerate(catalog, start=1):
    print(f"{i}. {product}")

# ============================
# Save Output
# ============================

catalog_df = pd.DataFrame({
    "UniqueProduct": catalog
})

catalog_df.to_csv(
    "Unique_Product_Catalog.csv",
    index=False
)

print("\nCatalog Saved Successfully!")

INPUT DATASET
Apple iPhone 15
iPhone 15 Smartphone
Samsung Galaxy S24
Galaxy S24 Mobile
HP Pavilion Laptop
HP Pavilion Notebook
Dell Wireless Mouse
Dell Optical Mouse
Sony WH-1000XM5 Headphones
Sony Wireless Headphones
Canon EOS 200D Camera
Canon DSLR Camera
Boat Rockerz 450 Headphones
Boat Wireless Headphones
Nike Air Max Shoes
Nike Running Shoes
Adidas Sports T-Shirt
Adidas Athletic Tee
Wildcraft Travel Backpack
Wildcraft Laptop Backpack
Milton Water Bottle
Milton Steel Bottle
Prestige Pressure Cooker
Prestige 5L Cooker
Pigeon Non-Stick Pan
Pigeon Frying Pan
LG 260L Refrigerator
LG Double Door Fridge
Whirlpool Washing Machine
Whirlpool Top Load Washer


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.




DUPLICATE PRODUCT GROUPS

Group 1
----------------------------------------
Apple iPhone 15
iPhone 15 Smartphone

Representative Product:
Apple iPhone 15

Group 2
----------------------------------------
Samsung Galaxy S24
Galaxy S24 Mobile

Representative Product:
Samsung Galaxy S24

Group 3
----------------------------------------
HP Pavilion Laptop
HP Pavilion Notebook

Representative Product:
HP Pavilion Laptop

Group 4
----------------------------------------
Dell Wireless Mouse
Dell Optical Mouse

Representative Product:
Dell Wireless Mouse

Group 5
----------------------------------------
Sony WH-1000XM5 Headphones
Sony Wireless Headphones

Representative Product:
Sony WH-1000XM5 Headphones

Group 6
----------------------------------------
Nike Air Max Shoes
Nike Running Shoes

Representative Product:
Nike Air Max Shoes

Group 7
----------------------------------------
Adidas Sports T-Shirt
Adidas Athletic Tee

Representative Product:
Adidas Sports T-Shirt

Group 8
------------